**How would we translate a query like this?**
``` sql
SELECT * 
FROM stuff
WHERE a=7
```

**Create some fake data**


In [1]:
from faker import Faker

fake = Faker()
fake.seed_instance(42)

from dataclasses import dataclass

number_of_tuples = 40


@dataclass
class Stuff:
    a: int
    b: int


# create some fake data mimicking tuples in a relation:
data: list[Stuff] = [
    Stuff(fake.pyint(max_value=20), fake.pyint(max_value=42))
    for f in range(number_of_tuples)
]

data[:10]

[Stuff(a=20, b=7),
 Stuff(a=0, b=17),
 Stuff(a=7, b=14),
 Stuff(a=4, b=6),
 Stuff(a=17, b=5),
 Stuff(a=18, b=27),
 Stuff(a=1, b=1),
 Stuff(a=2, b=13),
 Stuff(a=7, b=32),
 Stuff(a=19, b=1)]

**Algorithm 1: Scan per Query**

In [2]:
def scan(data: list[Stuff], attribute, constant) -> list[Stuff]:
    """Scans a list of Stuff-objects and returns all objects where the attribute is equal to the constant.
    :param data: the list of Stuff-objects
    :param attribute: the attribute to check
    :param constant: the constant to check against
    :return: a list of Stuff-objects
    """

    res: list[Stuff] = list[Stuff]()
    el: Stuff
    for el in data:
        if el.__dict__[attribute] == constant:
            res.append(el)
    return res


scan(data, "a", 7)

[Stuff(a=7, b=14), Stuff(a=7, b=32), Stuff(a=7, b=18)]


**What if we have more queries?**
``` sql
SELECT * 
FROM stuff
WHERE a=7
```
``` sql
SELECT * 
FROM stuff
WHERE b=32
```
``` sql
SELECT * 
FROM stuff
WHERE b=1
```


**Algorithm 2: One scan()-call per query**

In [3]:
from typing import Dict

# a dictionary with the queries we want to run:
queries_dict: Dict[str, Dict[str, str | int]] = {
    "q1": {"attribute": "a", "constant": 7},
    "q2": {"attribute": "b", "constant": 32},
    "q3": {"attribute": "b", "constant": 1},
}

key: str
value: Dict[str, str]

# run each query with a separate call to scan() for each query:
for key, value in queries_dict.items():
    print(key, scan(data, value["attribute"], value["constant"]))

q1 [Stuff(a=7, b=14), Stuff(a=7, b=32), Stuff(a=7, b=18)]
q2 [Stuff(a=7, b=32)]
q3 [Stuff(a=1, b=1), Stuff(a=19, b=1)]


**Algorithm 3: Shared Scan with separate lists for each of the queries**

In [4]:
def shared_scan(
    data: list[Stuff], queries_dict: Dict[str, Dict[str, str]]
) -> dict[str, list[Stuff]]:
    """Shared scan with result lists. Scans a list of Stuff-objects and returns a dictionary with the query as key and for each
    query the list of results as value."""

    # create a result dictionary with one result list per incoming query:
    res: dict[str, list[Stuff]] = {k: [] for k in queries_dict}

    # loop over all tuples:
    el: Stuff
    for el in data:
        query: str
        entry: Dict[str, str]
        # check for each query...
        for query, entry in queries_dict.items():
            # whether this tuple is part of the result:
            if el.__dict__[entry["attribute"]] == entry["constant"]:
                # append to dictionary:
                res[query].append(el)

    return res


shared_scan(data, queries_dict)

{'q1': [Stuff(a=7, b=14), Stuff(a=7, b=32), Stuff(a=7, b=18)],
 'q2': [Stuff(a=7, b=32)],
 'q3': [Stuff(a=1, b=1), Stuff(a=19, b=1)]}

**Algorithm 4: Shared Scan with a single list and tagging**

In [5]:
@dataclass
class Result:
    """A result class that holds the result of a query and a list of query names (tags) indicating whether the tuple is relevant for the query (part of the query result)."""

    result: Stuff
    relevant_for_queries: list[str]


def shared_scan_with_tagging(
    data: list[Stuff], queries_dict: Dict[str, Dict[str, str]]
) -> list[Result]:
    """Shared scan using tagging. Creates a list of Result-objects, each element is tagged with a list of query names indicating
    whether the tuple is relevant for the query."""

    res = list[Result]()

    # loop over all tuples:
    el: Stuff
    for el in data:
        # a white list of query names indicating whether the tuple is relevant for the query:
        relevant_for_queries: list[str] = list[str]()

        # check predicate for each query...
        query: str
        entry: Dict[str, str]
        for query, entry in queries_dict.items():
            # check whether this tuple is part of the result and append True or False to bit list:
            if el.__dict__[entry["attribute"]] == entry["constant"]:
                relevant_for_queries.append(query)

        # only append anything to the result list if at least one query has the el as its result:
        if len(relevant_for_queries) > 0:
            res.append(Result(el, relevant_for_queries))

    return res


def join_result_list_with_bit_lists(
    tagged_result: list[Result],
) -> dict[str, list[Stuff]]:
    """Builds a dictionary with the query as key and the list of results as value.
    This is required to be able to compare the results of the different algorithms in the unit tests.
    :param result: the list of Result-objects
    :return: a dictionary with the query as key and the list of results as value
    """

    result: dict[str, list[Stuff]] = dict[str, list[Stuff]]()
    item: Result
    for item in tagged_result:
        query: str
        for query in item.relevant_for_queries:
            result.setdefault(query, list[Stuff]()).append(item.result)

    return result


# run the algorithm
print("Result of shared scan with tagging:")
result: list[Result] = shared_scan_with_tagging(data, queries_dict)
for r in result:
    print(r)

# convert the result to a dictionary with the query as key and the list of results as value (as above):
print("\nconverted dict:")
result_as_dict: dict[str, list[object]] = join_result_list_with_bit_lists(result)

import pprint

pp = pprint.PrettyPrinter(indent=4)
pp.pprint(result_as_dict)

Result of shared scan with tagging:
Result(result=Stuff(a=7, b=14), relevant_for_queries=['q1'])
Result(result=Stuff(a=1, b=1), relevant_for_queries=['q3'])
Result(result=Stuff(a=7, b=32), relevant_for_queries=['q1', 'q2'])
Result(result=Stuff(a=19, b=1), relevant_for_queries=['q3'])
Result(result=Stuff(a=7, b=18), relevant_for_queries=['q1'])

converted dict:
{   'q1': [Stuff(a=7, b=14), Stuff(a=7, b=32), Stuff(a=7, b=18)],
    'q2': [Stuff(a=7, b=32)],
    'q3': [Stuff(a=1, b=1), Stuff(a=19, b=1)]}


**Unit Tests**

In [6]:
import unittest


class MyTest(unittest.TestCase):

    # the following function will be called before every test-function:

    def create_data(self, number_of_tuples: int):
        self.data: list[Stuff] = [
            Stuff(fake.pyint(max_value=20), fake.pyint(max_value=20))
            for f in range(number_of_tuples)
        ]

    def test_scans_end_to_end(self):
        i: int
        for i in range(10):
            self.create_data(2000)

            # a dictionary with queries
            _queries_dict: dict[str, dict[str, str | int]] = {
                "q"
                + str(i): {
                    "attribute": "a" if fake.pybool() else "b",
                    "constant": fake.pyint(max_value=20),
                }
                for i in range(30)
            }
            # compute the result for all these queries by calling the scan()-method for each query independently:
            res1_dict: dict[str, list[Stuff]] = {
                query: scan(self.data, entry["attribute"], entry["constant"])
                for query, entry in _queries_dict.items()
            }

            # single call to the shared_scan-method:
            res2_dict: dict[str, list[Stuff]] = shared_scan(self.data, _queries_dict)

            # single call to the shared_scan-method with tagging:
            res3_dict: dict[str, list[Stuff]] = join_result_list_with_bit_lists(
                shared_scan_with_tagging(self.data, _queries_dict)
            )

            # end to end comparison of the results:
            self.assertEqual(res1_dict, res2_dict)
            self.assertEqual(res1_dict, res3_dict)


# only execute a specific Test-class:
unittest.main(argv=["ignored", "-v"], defaultTest="MyTest", verbosity=2, exit=False)

test_scans_end_to_end (__main__.MyTest.test_scans_end_to_end) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.118s

OK
